In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import importlib 
m = importlib.import_module(".model", "src")
em = importlib.import_module(".em", "src")
from matplotlib import pyplot as plt
import numpy as np
import xgi
import scipy.special as ss

# Generate a synthetic hypergraph

In [3]:
# generate some sample data

eta   = 0.9   # node retention in edges
gamma = 0.2   # poisson number of nodes from graph
beta  = 0.7   # poisson number of novel nodes

timesteps = int(5e2)

H = m.GrowingHypergraph()
H.add_edge((0, 1))
H.add_edge((2, 3))

for _ in range(timesteps):
    H.sample_edge(eta, gamma, beta, True)

In [4]:
# estimator likelihood definitions
# pmfs are used in E-step
# m_step estimators are the parameter estimators for each distribution
# used in m-step, using the matrix chi formed in the E-step

# edge sampling
def edge_sample_pmf(x, t, eta):
    return (eta**x)*((1-eta)**(t-x))

def edge_sample_m_step(x, t, chi):
    C = np.tril(chi,-1)
    return (x*C).sum() / (t*C).sum()

esl = em.EdgeSampleLikelihood(
    pmf = edge_sample_pmf, 
    m_step = edge_sample_m_step
    )

# edge sampling: guaranteed version
def guaranteed_edge_sample_pmf(x, t, eta):
    M = (eta**(x-1))*((1-eta)**(t-x)) # ?
    M[x == 0] = 0
    return M

def guaranteed_edge_sample_m_step(x, t, chi):
    C = np.tril(chi,-1)
    top = ((x-1)*C).sum()
    bottom = ((t-1)*C).sum()
    return top / bottom

eslg = em.EdgeSampleLikelihood(
    pmf = guaranteed_edge_sample_pmf, 
    m_step = guaranteed_edge_sample_m_step
    )

# addition of novel nodes not previously seen in the hypergraph
def novel_nodes_pmf(k, beta):
     return (beta**k)*np.exp(beta)/(ss.factorial(k))

def novel_nodes_m_step(k, chi):
    return k.mean()

nnl = em.NovelNodesLikelihood(pmf = novel_nodes_pmf, m_step = novel_nodes_m_step)

# addition of nodes from rest of hypergraph
def nodes_from_hypergraph_pmf(k, gamma):
    return (gamma**k)*np.exp(-gamma)/(ss.factorial(k))

def nodes_from_hypergraph_m_step(x, chi):
    return (np.tril(chi, -1)*x).sum(axis = 1).mean()

nhl = em.NodesFromHypergraphLikelihood(pmf = nodes_from_hypergraph_pmf, m_step = nodes_from_hypergraph_m_step)

In [5]:
# now we build a model using the guaranteed version of the sampler 
# because that's how we built the original hypergraph

model = em.EM(eslg, nhl, nnl) 

# haven't fully implemented a clean EM algorithm so we need to step through the individual components: first form all the relevant arrays: 
model.form_arrays(H)

# then initialize (intentionally very bad guess)
model.pars = {"eta" : 0.1, "beta" : 0.1, "gamma" : 0.7}


In [6]:
# then run this cell until convergence
for _ in range(20):
    model.E_step()
    model.M_step()
model.pars

{'eta': 0.899065741607148,
 'beta': 0.7370517928286853,
 'gamma': 0.2037678188023633}